# Helping EVs leave Mariposa

Imagine a wildfire is approaching the community represented by **node 4 in Mariposa County**. You are helping the evacuation team plan for **600 electric vehicles**. People need to know when to leave, where they can charge, and how to reach safety with enough energy to finish the journey.

For this hypothetical scenario, **node 4 is the threatened origin**. **Nodes 1 and 10 are assumed safe destinations**, with 300 EVs assigned to each. These are planning assumptions for the supplied case, not information about an actual incident. Fire spread, road closures, and a hazard arrival deadline are not modeled.

### Start with the place

The supplied network has 12 nodes and 34 directed road connections. Follow the red triangle at node 4 toward the green destination squares. Blue **C** markers show fixed charging stations; dashed amber circles mark places where an MCS may be stationed.

![Mariposa scenario: threatened origin at node 4, safe destinations at nodes 1 and 10, fixed chargers at nodes 4, 6, and 9, and eligible MCS sites.](assets/mariposa_scenario.svg)

The figure uses the supplied local node coordinates. It shows the assumed roles of locations, not a wildfire boundary or a county-wide hazard map.

### What charging help is already there?

There are **20 fixed charging ports at the origin, node 4**, another **20 at node 6**, and **40 at node 9**. Each port provides 40 kW. In the starting case, every EV has a 30 kWh battery at 20% state of charge: 6 kWh before departure. Charging is therefore part of the evacuation plan, alongside routes and departure times.

For our first plan, **one mobile charging station is available**. It carries 420 kWh and has five 40 kW ports. We can place it at an eligible site before the evacuation; it stays there throughout the run. The input inventory contains 15 units, so we can later explore making more units available. A dashed circle on the map marks an option, not a site that has already been selected.

The planning question is now concrete: **where should that one MCS go, and how should the vehicles depart and charge?** We will choose what “finishing sooner” means, solve for a plan, replay it, and look at the trips and deployment it produces.

### Run the plan

Follow the [README](../README.md) to install the package and configure your solver, then select the **EVac** kernel. Run cells from top to bottom with Shift+Enter. Sections 1–6 produce and save a plan; section 7 offers a few comparisons.

Mariposa exceeds Gurobi's free restricted model size. Use a suitable Gurobi license as described in the README. The startup check below confirms that the solver can start; model-size restrictions are checked when solving. The supplied inputs and finite path library define this example, which does not reproduce the paper's full numerical study.


In [ ]:
from pathlib import Path
from datetime import datetime
from time import perf_counter
import pandas as pd
from IPython.display import SVG, display
from evac import (load_scenario, prepare, MILP,
                  plan, simulate, evaluate, RunResult, RunCase, run,
                  probe_gurobi,
                  save_result, save_prepared_scenario)
from evac.reporting import from_prepared, from_result
from evac.visualization import map_svg

# Locate bundled inputs from the project root or example folder.
ROOT = Path.cwd()
if ROOT.name == "example":
    ROOT = ROOT.parent
if not (ROOT / "data" / "scenarios").is_dir():
    raise FileNotFoundError("Open this notebook from the project root or example folder.")


## 1. Choose what the plan should achieve

Start with one MCS and 20% initial SOC. Our first objective is `"mean"`: bring down the average completion time across all 600 vehicles. Choosing `"max"` instead puts the last vehicle's completion time first. The choice changes the optimization itself, so make it before solving.

Keep the supplied settings for the first run. After changing them, restart the kernel and rerun from the top. If the solver check fails, resolve the reported installation or license issue before continuing.


In [ ]:
MCS_COUNT = 1  # Available mobile units: integers 0–15 from the supplied inventory.
INITIAL_SOC = 0.20  # Initial EV state of charge as a fraction in [0, 1].
OBJECTIVE = "mean"  # "mean" minimizes average completion; "max" minimizes the last completion.
TIME_LIMIT_SECONDS = 180  # Positive solver time limit; None removes the time limit.
THREADS = 1  # Positive worker count; keep the same count across comparisons.
SEED = 7  # Nonnegative solver seed; keep it fixed across comparisons.

# Check solver availability before preparing the larger model.
probe = probe_gurobi(runtime=True)
display(pd.DataFrame([{"installed": probe.dependency_installed,
                       "runtime_usable": probe.runtime_usable,
                       "reason": probe.reason}]))
if not probe.runtime_usable:
    raise RuntimeError(probe.reason)

scenario = load_scenario(ROOT / "data/scenarios/mariposa.yaml")
solver = MILP(time_limit_seconds=TIME_LIMIT_SECONDS, threads=THREADS, seed=SEED)
# Keep earlier results by creating a new folder for each run.
output_dir = ROOT / "results" / ("mariposa_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
output_dir.mkdir(parents=True, exist_ok=False)
print("This run's output folder:", output_dir.relative_to(ROOT))

Keep solver settings fixed when comparing physical inputs or objectives. `TIME_LIMIT_SECONDS` covers optimization; preparation, model construction, simulation, and saving take additional time. The first simulation also compiles numerical kernels.

## 2. Turn the situation into model inputs

Now let `prepare` check the inputs, apply your choices, and build the candidate routes and charging options. The counts below confirm how many vehicles and MCS units the plan will use. The opening map depicts the supplied geography; the map after solving will show which roads the plan uses.


In [ ]:
# Time input preparation separately from optimization.
started = perf_counter()
prepared = prepare(scenario, objective=OBJECTIVE, mcs_count=MCS_COUNT, initial_soc=INITIAL_SOC)
preparation_seconds = perf_counter() - started
print(f"{len(prepared.vehicles)} vehicles; {len(prepared.mcs_units)} MCS units; "
      f"{len(prepared.candidate_paths)} retained charging paths")
print("Objective passed to the MILP:", prepared.evaluation.objectives[0].norm)
map_views = from_prepared(prepared)


The plan works within a few concrete rules. Charging and MCS discharge efficiencies are each 0.9. A 300-second charging increment delivers 3 kWh to an EV, or 10% of its battery. Vehicles may depart on a 150-second grid, up to 7,050 seconds after the common evacuation start; they may finish later.

Road travel times stay fixed in this example. The MCS location also stays fixed. These assumptions let us focus on coordinating limited charging capacity with the vehicles' departures.


## 3. Find a deployment and departure plan

We have the vehicles, charging resources, and a chosen objective. `plan` now asks the MILP solver to coordinate the MCS location, routes, departures, and charging. Check whether it found a plan and whether it proved optimality before interpreting the objective.


In [ ]:
planner = plan(prepared, solver)
# These objective values and bounds describe the MILP before simulation.
display(pd.Series({
    "solver_status": planner.termination.value,
    "has_plan": planner.plan is not None,
    "proven_optimal": planner.termination.value == "optimal",
    "formulation_objective_seconds": planner.formulation_objective,
    "formulation_bound_seconds": planner.objective_bound,
    "relative_gap": planner.objective_gap,
    "terminal_cause": planner.terminal_cause,
}, name="value").to_frame())


`optimal` means the selected finite MILP and its objective hierarchy were solved to the solver's tolerances. `feasible` means a plan was found but the complete solve did not prove optimality. `infeasible` means the finite model has no feasible plan. `no_incumbent` means no plan was returned, for example after a short time limit.

The formulation objective is expressed in seconds: for `mean`, the raw sum of vehicle completion times is divided by the number of vehicles; for `max`, it is the completion-time envelope. Raw accepted solver values and objective certificates are preserved in the saved result.

## 4. Follow the vehicles through the plan

A proposed schedule is only part of the picture. Replay it with `simulate`, then use `evaluate` to see when the vehicles actually finish in the simulation. This gives both mean and maximum completion for the same plan. If the solver returned no plan, the cell keeps its status and skips the replay.


In [ ]:
# Without a plan, simulation and evaluation results stay unavailable.
simulation = None
evaluation = None
simulation_seconds = 0.0
evaluation_seconds = 0.0
if planner.plan is not None:
    started = perf_counter()
    simulation = simulate(prepared, planner.plan)
    simulation_seconds = perf_counter() - started
    started = perf_counter()
    # Compute both mean and max for the same simulated plan.
    evaluation = evaluate(prepared, simulation)
    evaluation_seconds = perf_counter() - started
else:
    print("No plan is available for simulation. Check the solver status and termination reason above.")

# Collect the stages and their timings for reporting and saving.
baseline = RunResult(planner, simulation, evaluation, timings={
    "preparation_seconds": preparation_seconds,
    "build_seconds": planner.metadata["build_seconds"],
    "solve_seconds": planner.metadata["solve_seconds"],
    "simulation_seconds": simulation_seconds,
    "evaluation_seconds": evaluation_seconds,
})
views = from_result(baseline)
# Show the key results here; section 6 saves the full summary and event tables.
summary_fields = [
    "solver_status", "has_plan", "proven_optimal", "simulation_feasible",
    "finished_vehicle_count", "simulated_objective_seconds",
    "mean_completion_seconds", "max_completion_seconds",
    "mean_evacuation_duration_seconds", "max_evacuation_duration_seconds",
    "preparation_seconds", "build_seconds", "solve_seconds",
    "simulation_seconds", "evaluation_seconds",
]
summary = views.table("summary").to_dataframe().set_index("metric")
display(summary.reindex(summary_fields))

Read both simulated mean and max for this same plan. Completion time is measured from the common evacuation start (time zero). Evacuation duration is measured from each vehicle's actual departure; it excludes waiting at the origin before departure. Solve and simulation seconds measure computer time, not evacuation time.

A feasible MILP solution and a successful replay are separate checks. If `simulation_feasible` is false, record the failure and do not treat unavailable mean/max values as zero.

## 5. See where help goes and when vehicles leave

First look at the MCS deployment: which eligible site did the solver choose? Then read the departure and charging schedule. The preview shows 12 vehicles; section 6 saves all 600. The map keeps the origin, destinations, and chargers visible while highlighting roads used by the returned plan. Dashed circles still show eligible sites; the deployment table identifies the selected one.


In [ ]:
display(views.table("mcs_deployments").to_dataframe())
schedule = views.table("schedule").to_dataframe()
display(schedule.head(12))
print("The full vehicle schedule is saved as a CSV in the next cell.")
# Highlight the roads used by the returned plan.
if baseline.plan is not None:
    display(SVG(map_svg(map_views.table("map_nodes"), map_views.table("map_edges"),
                       path_edges=map_views.table("candidate_path_edges"),
                       route_usage=views.table("route_usage"), title="The evacuation plan on the Mariposa network",
                       origin_label="Threatened origin", destination_label="Safe destination")))


## 6. Keep the plan and its results

Save the prepared inputs, full result, and reporting tables together. The folder below contains the complete vehicle schedule, MCS deployment, and metrics. Keeping `prepared.yaml` with `result.yaml` lets you return to the same plan and replay it later.

You can stop here with a saved run. The [API guide](../docs/python_api.md#reload-a-saved-run) shows how to reload it.


In [ ]:
baseline_dir = output_dir / "baseline"
# Save matching inputs so the stored plan can be simulated again.
save_prepared_scenario(prepared, baseline_dir / "prepared.yaml")
save_result(baseline, baseline_dir / "result.yaml")
views.save_csv(baseline_dir / "tables")
print("Results:", baseline_dir.relative_to(ROOT))


## 7. What if the team had different resources?


The first plan gives us a reference point. Would two additional MCS units help? What if vehicles started with a little more charge? And what changes when the priority is the last vehicle to arrive?

Set `RUN_COMPARISONS = True` below to explore three independent changes:

1. Make **3 MCS units** available instead of 1.
2. Raise initial SOC from **0.20 to 0.25**.
3. Change the objective from **`"mean"` to `"max"`**.

Adjust `comparison_settings` if you changed the baseline. Each variant is saved separately. `comparison.csv` retains all summary metrics; the on-screen table shows the main results. Read the solver status and simulation feasibility alongside each comparison.


In [ ]:
RUN_COMPARISONS = False  # True runs all three variants; False keeps only the baseline.

# Each tuple sets: run name, MCS count, initial SOC, and objective.
comparison_settings = [
    ("more_mcs", 3, INITIAL_SOC, OBJECTIVE),
    ("higher_soc", MCS_COUNT, 0.25, OBJECTIVE),
    ("max_objective", MCS_COUNT, INITIAL_SOC, "max"),
]
results = {"baseline": baseline}
if RUN_COMPARISONS:
    # Use the same solver settings for each independent case.
    for name, mcs_count, initial_soc, objective in comparison_settings:
        print("Running", name)
        case = RunCase(scenario, solver, objective, mcs_count, initial_soc)
        results[name] = run(case, output_dir=output_dir / name)
        from_result(results[name]).save_csv(output_dir / name / "tables")

    # One row per run; save every summary field before selecting display columns.
    comparison = pd.DataFrame({
        name: dict(from_result(result).table("summary").rows)
        for name, result in results.items()
    }).T
    comparison.to_csv(output_dir / "comparison.csv", index_label="experiment")
    columns = ["solver_status", "proven_optimal", "simulation_feasible",
               "mean_completion_seconds", "max_completion_seconds", "solve_seconds"]
    display(comparison.reindex(columns=columns))
    for name, result in results.items():
        print(name)
        display(from_result(result).table("mcs_deployments").to_dataframe())
else:
    print("Comparisons skipped. Set RUN_COMPARISONS = True to run this section.")


### Watch the evacuation finish

The curves show vehicles reaching their destinations as time passes. Look at the earlier arrivals as well as the last one: a plan that improves the average may look different from one that focuses on the final vehicle. The figure is saved as `completion_curves.png`.


In [ ]:
if RUN_COMPARISONS:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4))
    plotted = False
    for name, result in results.items():
        curve = from_result(result).table("evacuation_completion_series").to_dataframe()
        # Skip runs without completion data.
        if not curve.empty:
            # Convert simulated seconds to minutes for the horizontal axis.
            ax.step(curve["time_seconds"] / 60, curve["arrived_vehicle_count"], where="post", label=name)
            plotted = True
    ax.set(xlabel="Minutes from evacuation start", ylabel="Vehicles at destinations",
           title="Simulated evacuation completion")
    if plotted:
        ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(output_dir / "completion_curves.png", dpi=160)
    plt.show()

Compare mean and maximum completion for each plan: improving one measure can increase the other. Check the deployment tables for changes in MCS locations or counts. Mean completion minus mean evacuation duration gives the average wait before departure.

Small differences in computation time can reflect other programs, first-use compilation, or solver versions. Multiple schedules can attain the same objective, so a deployment change alone does not establish an improvement.